# Dataset Reconnaissance
Inspect the structure of each mounted dataset **without extracting**.
Run this once, push results to HuggingFace, then never re-run.

In [ ]:
# Cell 1: Install deps (run this FIRST, before imports)
!pip install -q rarfile huggingface_hub
print("Install done. Run the next cell.")

In [ ]:
# Cell 2: Imports (after install)
import os, json, zipfile, rarfile
from pathlib import Path
from collections import Counter, defaultdict

REPO_ID = "david-net-av/backup"  # HF repo for recon reports
KAGGLE_INPUT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/dataset_recon")
OUT_DIR.mkdir(exist_ok=True)

print("Setup done.")

In [ ]:
# Cell 3: Discover mounted datasets
# Handles two cases:
#   A) Individual datasets: /kaggle/input/fakeavceleb/, /kaggle/input/lav-df/, ...
#   B) Single "datasets" folder: /kaggle/input/datasets/fakeavceleb/, ...

mounted = []
for d in sorted(KAGGLE_INPUT.iterdir()):
    if not d.is_dir():
        continue
    # Check if this is a wrapper folder containing the actual datasets
    subdirs = [s for s in d.iterdir() if s.is_dir()]
    if d.name == "datasets" and len(subdirs) > 1:
        # Case B: datasets/ is a wrapper, real datasets are inside
        print(f"{d.name}/ is a wrapper, checking inside...")
        for s in sorted(subdirs):
            mounted.append(s)
            print(f"  {s.name}/")
    else:
        # Case A: this is a direct dataset mount
        mounted.append(d)
        print(f"  {d.name}/")

print(f"\nFound {len(mounted)} datasets.")

In [ ]:
# Cell 4: Recon functions
def recon_zip(path: Path) -> dict:
    """List contents of a zip without extracting."""
    try:
        with zipfile.ZipFile(path) as zf:
            entries = []
            for info in zf.infolist():
                entries.append({
                    "name": info.filename,
                    "size": info.file_size,
                    "compressed": info.compress_size,
                })
            return {"type": "zip", "entries": entries, "count": len(entries)}
    except Exception as e:
        return {"type": "zip", "error": str(e)}


def recon_rar(path: Path) -> dict:
    """List contents of a rar without extracting."""
    try:
        rf = rarfile.RarFile(path)
        entries = []
        for info in rf.infolist():
            entries.append({
                "name": info.filename,
                "size": info.file_size,
                "compressed": info.compress_size if hasattr(info, 'compress_size') else 0,
            })
        return {"type": "rar", "entries": entries, "count": len(entries)}
    except Exception as e:
        return {"type": "rar", "error": str(e)}


def build_tree(entries: list[dict]) -> dict:
    """Build directory tree from flat file list."""
    tree = {}
    exts = Counter()
    total_size = 0
    for e in entries:
        name = e["name"]
        size = e.get("size", 0)
        total_size += size
        ext = Path(name).suffix.lower()
        exts[ext] += 1
        parts = name.split("/")
        node = tree
        for p in parts[:-1]:
            if p not in node:
                node[p] = {}
            node = node[p]
    return {"tree": tree, "extensions": dict(exts), "total_size": total_size, "file_count": len(entries)}


def recon_archive(path: Path) -> dict:
    """Dispatch to zip or rar recon."""
    ext = path.suffix.lower()
    if ext == ".zip":
        return recon_zip(path)
    elif ext in (".rar", ".001"):
        return recon_rar(path)
    else:
        return {"type": "unknown", "error": f"unsupported extension: {ext}"}


print("Recon functions ready.")

In [ ]:
# Cell 5: Run recon on all mounted datasets
results = {}

for ds_dir in mounted:
    ds_name = ds_dir.name
    print(f"\n=== {ds_name} ===")
    
    # Scan for archives and loose files
    archives = []
    loose_files = []
    try:
        for f in ds_dir.rglob("*"):
            if f.is_file():
                # Detect archives: .zip, .rar, or multi-part (.001, .002, etc.)
                is_archive = (
                    f.suffix.lower() in (".zip", ".rar") or
                    (f.suffix.isdigit() and len(f.suffix) == 3 and f.suffix != "000")
                )
                if is_archive:
                    archives.append(f)
                else:
                    loose_files.append({"name": str(f.relative_to(ds_dir)), "size": f.stat().st_size})
    except Exception as e:
        print(f"  Error scanning: {e}")
        continue
    
    report = {
        "name": ds_name,
        "path": str(ds_dir),
        "archives": [],
        "loose_files_count": len(loose_files),
        "loose_files_sample": loose_files[:20],
    }
    
    # Analyze archives (limit to 5 to save time)
    for arch in sorted(archives, key=lambda p: p.name)[:5]:
        print(f"  Scanning: {arch.name} ({arch.stat().st_size / 1e6:.1f} MB)")
        recon = recon_archive(arch)
        if "entries" in recon:
            tree_info = build_tree(recon["entries"])
            recon["tree"] = tree_info["tree"]
            recon["extensions"] = tree_info["extensions"]
            recon["total_size"] = tree_info["total_size"]
            recon["file_count"] = tree_info["file_count"]
            print(f"    -> {tree_info['file_count']} files, {tree_info['total_size'] / 1e9:.2f} GB")
            print(f"    Extensions: {tree_info['extensions']}")
        else:
            print(f"    -> ERROR: {recon.get('error', 'unknown')}")
        report["archives"].append({"name": arch.name, "size": arch.stat().st_size, "recon": recon})
    
    # If no archives, show directory structure from loose files
    if not archives and loose_files:
        print(f"  No archives found. Analyzing {len(loose_files)} loose files...")
        top_dirs = Counter()
        exts = Counter()
        total_size = 0
        for lf in loose_files:
            parts = lf["name"].split("/")
            if len(parts) >= 2:
                top_dirs[parts[0]] += 1
            ext = Path(lf["name"]).suffix.lower()
            exts[ext] += 1
            total_size += lf["size"]
        print(f"  Top-level dirs: {dict(top_dirs)}")
        print(f"  Extensions: {dict(exts)}")
        print(f"  Total size: {total_size / 1e9:.2f} GB")
        report["top_dirs"] = dict(top_dirs)
        report["extensions"] = dict(exts)
        report["total_size"] = total_size
    
    results[ds_name] = report
    print(f"  Archives: {len(archives)}, Loose files: {len(loose_files)}")

print(f"\nRecon complete for {len(results)} datasets.")

In [ ]:
# Cell 6: Save reports
for name, report in results.items():
    out_path = OUT_DIR / f"{name}.json"
    with open(out_path, "w") as f:
        json.dump(report, f, indent=2, default=str)
    print(f"Saved: {out_path}")

# Summary
print("\n=== SUMMARY ===")
for name, report in results.items():
    total_archives = sum(a["recon"].get("total_size", 0) for a in report["archives"] if "recon" in a)
    print(f"{name}: {len(report['archives'])} archives, {total_archives/1e9:.2f} GB compressed, {report['loose_files_count']} loose files")

In [ ]:
# Cell 7: Push to HuggingFace
from huggingface_hub import HfApi

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("hf")
if not hf_token:
    print("ERROR: HF_TOKEN not found in Kaggle Secrets or env")
    print("Add it: Settings -> Secrets -> Add -> Name=HF_TOKEN")
else:
    api = HfApi(token=hf_token)
    try:
        api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
    except Exception as e:
        print(f"Repo note: {e}")
    
    for f in OUT_DIR.glob("*.json"):
        api.upload_file(
            path_or_fileobj=str(f),
            path_in_repo=f"dataset_recon/{f.name}",
            repo_id=REPO_ID,
            repo_type="model",
        )
        print(f"Pushed: {f.name}")
    
    print(f"\nAll reports pushed to {REPO_ID}/dataset_recon/")